In [ ]:
#!/usr/bin/env python3.92
# -*- coding: utf-8 -*-

import os 

import numpy as np
import csv, json
import pandas as pd

################################################################################################
## Preparing DJIA data
# Reading DJIA index prices csv file
UPDATE=True
PresentDate="2022-03-07"
# if (UPDATE):
#     from yahoo_fin.stock_info import get_data
#     df_vix = get_data("^VIX", start_date="01/01/11", end_date=PresentDate, index_as_date = True, interval="1d")
#     df_vix.to_csv("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/csv/vix.csv")
with open( "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/csv/vix.csv", 'r') as csvfile:
    spamreader = csv.reader(csvfile, delimiter=',')
    # Converting the csv file reader to a lists 
    data_list = list(spamreader)

# Separating header from the data
header = data_list[0] 
data_list = data_list[1:] 

data_list = np.asarray(data_list)

# Selecting date and close value for each day
selected_data = data_list[:, [0, 4, 5]]



df = pd.DataFrame(data=selected_data[0:,1:],
             index=selected_data[0:,0],
                                columns=['close', 'adj close'],
                                        dtype='float64')

# Reference for pandas interpolation http://pandas.pydata.org/pandas-docs/stable/missing_data.html
# Adding missing dates to the dataframe
df1 = df
idx = pd.date_range('01-01-2022', '31-01-2022')
df1.index = pd.DatetimeIndex(df1.index)
df1 = df1.reindex(idx, fill_value=np.NaN)
# df1.count() # gives 2518 count
interpolated_df = df1.interpolate()
interpolated_df.count() # gives 3651 count

# Removing extra date rows added in data for calculating interpolation
interpolated_df = interpolated_df[3:]

###############################################################################################  
## Preparing NYTimes data
# Function to parse and convert date format
date_format = ["%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%dT%H:%M:%S+%f"]
def try_parsing_date(text):
    for fmt in date_format:
        #return datetime.strptime(text, fmt)
        try:
            return datetime.strptime(text, fmt).strftime('%Y-%m-%d')
        except ValueError:
            pass
    raise ValueError('no valid date format found')




# years = [2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011]
# months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
years = [2022]
months = [1]
dict_keys = ['pub_date', 'headline'] #, 'lead_paragraph']
articles_dict = dict.fromkeys(dict_keys)
# Filtering list for type_of_material
type_of_material_list = ['blog', 'brief', 'news', 'editorial', 'op-ed', 'list','analysis']
# Filtering list for section_name
section_name_list = ['business', 'national', 'world', 'u.s.' , 'politics', 'opinion', 'tech', 'science',  'health']
news_desk_list = ['business', 'national', 'world', 'u.s.' , 'politics', 'opinion', 'tech', 'science',  'health', 'foreign']

current_date = '2022-01-31'
from datetime import datetime
#years = [2015]
#months = [3]

current_article_str = ''      

## Adding article column to dataframe
interpolated_df["articles"] = ''
count_articles_filtered = 0
count_total_articles = 0
count_main_not_exist = 0               
count_unicode_error = 0     
count_attribute_error = 0   
for year in years:
    for month in months:
        file_str = '/Users/Farid Srouji/Documents/Research/master/NYTimes/data/txt/' + str(year) + '-' + '{:02}'.format(month) + '.json'
        print (file_str)
        if not (os.path.isfile(file_str)):
            print("Error file not found")
            continue

        with open(file_str, encoding="utf8") as data_file:    
            NYTimes_data = json.load(data_file)

        count_total_articles = count_total_articles + len(NYTimes_data["response"]["docs"][:])
        for i in range(len(NYTimes_data["response"]["docs"][:])):
            try:
                if any(substring in NYTimes_data["response"]["docs"][:][i]['type_of_material'].lower() for substring in type_of_material_list):
                    if any(substring in NYTimes_data["response"]["docs"][:][i]['section_name'].lower() for substring in section_name_list):
                        #count += 1
                        print("---------------------- 1---------------------")
                        count_articles_filtered += 1
                        #print 'i: ' + str(i)
                        articles_dict = { your_key: NYTimes_data["response"]["docs"][:][i][your_key] for your_key in dict_keys }
                        articles_dict['headline'] = articles_dict['headline']['main'] # Selecting just 'main' from headline
                        #articles_dict['headline'] = articles_dict['lead_paragraph'] # Selecting lead_paragraph
                        date = try_parsing_date(articles_dict['pub_date'])
                        print("pub_date="+articles_dict['pub_date'])
                        print("data="+date)
                        print("current_date="+current_date)
                        #print 'article_dict: ' + articles_dict['headline']
                        if date == current_date:
                            print("---------------------- 2.1---------------------")
                            print(articles_dict['headline'])
                            current_article_str = current_article_str + '. ' + articles_dict['headline']
                            print("---------------------- 2.2---------------------")
                        else:  
                            print("---------------------- 3.1 ---------------------")
                            # print(current_date)
                            # print(interpolated_df.at[current_date, 'articles'])
                            # print("---------------------- 3.2 ---------------------")

                            interpolated_df.at[current_date, 'articles'] = interpolated_df.at[current_date, 'articles'] + '. ' + current_article_str
                            current_date = date
                        
                            #interpolated_df.set_value(date, 'articles', current_article_str)
                            #print str(date) + current_article_str
                            current_article_str = articles_dict['headline']
                            print("---------------------- 3.2 ---------------------")

                        # For last condition in a year
                        if (date == current_date) and (i == len(NYTimes_data["response"]["docs"][:]) - 1): 
                            #interpolated_df.set_value(date, 'articles', current_article_str)   
                            interpolated_df.at[date, 'articles'] = current_article_str

             #Exception for section_name or type_of_material absent
            except AttributeError:
                #print 'attribute error'
                #print NYTimes_data["response"]["docs"][:][i]
                count_attribute_error += 1
                # If article matches news_desk_list if none section_name found
                try:
                    if any(substring in NYTimes_data["response"]["docs"][:][i]['news_desk'].lower() for substring in news_desk_list):
                            #count += 1
                            count_articles_filtered += 1
                            #print 'i: ' + str(i)
                            articles_dict = { your_key: NYTimes_data["response"]["docs"][:][i][your_key] for your_key in dict_keys }
                            articles_dict['headline'] = articles_dict['headline']['main'] # Selecting just 'main' from headline
                            #articles_dict['headline'] = articles_dict['lead_paragraph'] # Selecting lead_paragraph
                            date = try_parsing_date(articles_dict['pub_date'])
                            print(articles_dict)
                            #print 'article_dict: ' + articles_dict['headline']
                            if date == current_date:
                                current_article_str = current_article_str + '. ' + articles_dict['headline']
                                print("current_article_str="+current_article_str)
                            else:  
                                interpolated_df.at[current_date, 'articles'] = interpolated_df.loc[current_date, 'articles'] + '. ' + current_article_str
                                current_date = date
                                #interpolated_df.set_value(date, 'articles', current_article_str)
                                #print str(date) + current_article_str
                                current_article_str = articles_dict['headline']
                                print(interpolated_df.at[current_date, 'articles'])

                            # For last condition in a year
                            if (date == current_date) and (i == len(NYTimes_data["response"]["docs"][:]) - 1): 
                                print("++++++++++++++++++++Last condition++++++++++++++++")
                                interpolated_df.at[date, 'articles'] = current_article_str
                    print("Success")
                except AttributeError:
                    print("AttributeError")
                    pass
                pass
            except KeyError:
                print('key error')
                current_date = date
                #print NYTimes_data["response"]["docs"][:][i]
                count_main_not_exist += 1
                pass   
            except TypeError:
                print("type error")
                #print NYTimes_data["response"]["docs"][:][i]
                count_main_not_exist += 1
                pass

In [ ]:
interpolated_df

In [ ]:
print ("count_articles_filtered="+str(count_articles_filtered))
print ("count_total_articles="+str(count_total_articles))
print ("count_main_not_exist="+str(count_main_not_exist))
print ("count_unicode_error="+str(count_unicode_error))

In [ ]:
interpolated_df

In [ ]:

## Putting all articles if no section_name or news_desk not found
for date, row in interpolated_df.T.iteritems():   
    if len(interpolated_df.loc[date, 'articles']) <= 400:
        #print interpolated_df.loc[date, 'articles']
        #print date
        month = date.month
        year = date.year
        file_str = '/Users/Farid Srouji/Documents/Research/master/NYTimes/data/txt/' + str(year) + '-' + '{:02}'.format(month) + '.json'
        print (file_str)
        if not (os.path.isfile(file_str)):
            continue
        with open(file_str, encoding="utf8") as data_file:    
            NYTimes_data = json.load(data_file)
        count_total_articles = count_total_articles + len(NYTimes_data["response"]["docs"][:])
        #interpolated_df.set_value(date.strftime('%Y-%m-%d'), 'articles', '')
        interpolated_df.at[date.strftime('%Y-%m-%d'), 'articles'] = ''

        for i in range(len(NYTimes_data["response"]["docs"][:])):
            try:

                articles_dict = { your_key: NYTimes_data["response"]["docs"][:][i][your_key] for your_key in dict_keys }
                articles_dict['headline'] = articles_dict['headline']['main'] # Selecting just 'main' from headline
                #articles_dict['headline'] = articles_dict['lead_paragraph'] # Selecting lead_paragraph       
                pub_date = try_parsing_date(articles_dict['pub_date'])
                #print 'article_dict: ' + articles_dict['headline']
                if date.strftime('%Y-%m-%d') == pub_date: 
                    #interpolated_df.set_value(pub_date, 'articles', interpolated_df.loc[pub_date, 'articles'] + '. ' + articles_dict['headline'])
                    interpolated_df.at[pub_date, 'articles'] = interpolated_df.at[pub_date, 'articles'] + '. ' + articles_dict['headline']

            except KeyError:
                print('key error')
                #print NYTimes_data["response"]["docs"][:][i]
                #count_main_not_exist += 1
                pass   
            except TypeError:
                print("type error")
                #print NYTimes_data["response"]["docs"][:][i]
                #count_main_not_exist += 1
                pass

# Saving the data as pickle file
interpolated_df.to_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/pickled_one_year_filtered_lead_para_sp500_clean.pkl')  

# Save pandas frame in csv form
interpolated_df.to_csv('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/sample_interpolated_df_one_years_filtered_lead_para_sp500_clean.csv',
                    sep='\t', encoding='utf-8')

# Reading the data as pickle file
dataframe_read = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/pickled_one_year_filtered_lead_para_sp500_clean.pkl')

print("Finished")
#################################################################################